# 04 — Hyperparameter Tuning with Keras Tuner

Reference: [handson-ml3 ch11](https://github.com/ageron/handson-ml3/blob/main/11_training_deep_neural_networks.ipynb)

**Runtime → T4 GPU**

| Strategy | Description |
|---|---|
| `RandomSearch` | Random sampling of the search space |
| `Hyperband` | Early stopping of poor trials (efficient) |
| `BayesianOptimization` | Learns from past trials (smart search) |

**Hyperparameters tuned:** learning rate · dropout · L2 · num filters · dense units · num conv blocks |

In [ ]:
!pip install -q keras-tuner

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt
import os, shutil

print('TF:', tf.__version__)
print('Keras Tuner:', kt.__version__)

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0
y_train = y_train.flatten()
y_test  = y_test.flatten()

# Use a subset for faster tuning
x_val, y_val = x_test[:5000], y_test[:5000]

data_aug = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name='aug')

print(f'Train: {x_train.shape}  Val: {x_val.shape}')

In [ ]:
# ── HyperModel definition ────────────────────────────────────────────────────
def build_hypermodel(hp):
    """
    Hyperparameters explored:
      - num_conv_blocks : 2 or 3
      - filters_*       : 32 / 64 / 128 per block
      - dropout_conv    : 0.1 – 0.4
      - dense_units     : 128 / 256 / 512
      - dropout_dense   : 0.2 – 0.6
      - l2_reg          : 1e-4 – 1e-2 (log scale)
      - learning_rate   : 1e-4 – 1e-2 (log scale)
      - optimizer       : adam / sgd
    """
    inp = keras.Input(shape=(32, 32, 3))
    x   = data_aug(inp)

    num_blocks = hp.Int('num_conv_blocks', 2, 3)
    l2_val     = hp.Float('l2_reg', 1e-4, 1e-2, sampling='log')
    reg        = regularizers.l2(l2_val)

    for i in range(num_blocks):
        filters = hp.Choice(f'filters_{i}', [32, 64, 128])
        x = layers.Conv2D(filters, 3, padding='same', activation='relu',
                          kernel_regularizer=reg)(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(hp.Float(f'dropout_conv_{i}', 0.1, 0.4, step=0.1))(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(
            hp.Choice('dense_units', [128, 256, 512]),
            activation='relu',
            kernel_regularizer=reg
        )(x)
    x = layers.Dropout(hp.Float('dropout_dense', 0.2, 0.6, step=0.1))(x)
    out = layers.Dense(10, activation='softmax')(x)

    model = keras.Model(inp, out)

    lr = hp.Float('learning_rate', 1e-4, 1e-2, sampling='log')
    opt_name = hp.Choice('optimizer', ['adam', 'sgd'])
    if opt_name == 'adam':
        optimizer = keras.optimizers.Adam(lr)
    else:
        optimizer = keras.optimizers.SGD(lr, momentum=0.9, nesterov=True)

    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

print('HyperModel defined.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STRATEGY 1 — Random Search
# ══════════════════════════════════════════════════════════════════════════════
print('=== Random Search ===')
shutil.rmtree('/tmp/kt_random', ignore_errors=True)

tuner_random = kt.RandomSearch(
    build_hypermodel,
    objective='val_accuracy',
    max_trials=10,              # try 10 random configs
    executions_per_trial=1,
    directory='/tmp/kt_random',
    project_name='cifar10',
    overwrite=True
)

tuner_random.search_space_summary()

tuner_random.search(
    x_train, y_train,
    epochs=15,
    validation_data=(x_val, y_val),
    batch_size=128,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=0
)

print('\nBest hyperparameters (Random Search):')
best_hp_rs = tuner_random.get_best_hyperparameters(num_trials=1)[0]
for key, val in best_hp_rs.values.items():
    print(f'  {key}: {val}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STRATEGY 2 — Hyperband (efficient early stopping)
# ══════════════════════════════════════════════════════════════════════════════
print('=== Hyperband ===')
shutil.rmtree('/tmp/kt_hyperband', ignore_errors=True)

tuner_hb = kt.Hyperband(
    build_hypermodel,
    objective='val_accuracy',
    max_epochs=20,             # maximum epochs per trial
    factor=3,                  # reduction factor
    hyperband_iterations=1,
    directory='/tmp/kt_hyperband',
    project_name='cifar10',
    overwrite=True
)

tuner_hb.search(
    x_train, y_train,
    epochs=20,
    validation_data=(x_val, y_val),
    batch_size=128,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=0
)

print('\nBest hyperparameters (Hyperband):')
best_hp_hb = tuner_hb.get_best_hyperparameters(num_trials=1)[0]
for key, val in best_hp_hb.values.items():
    print(f'  {key}: {val}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STRATEGY 3 — Bayesian Optimisation
# ══════════════════════════════════════════════════════════════════════════════
print('=== Bayesian Optimisation ===')
shutil.rmtree('/tmp/kt_bayes', ignore_errors=True)

tuner_bayes = kt.BayesianOptimization(
    build_hypermodel,
    objective='val_accuracy',
    max_trials=10,
    num_initial_points=3,      # random seed trials before Bayesian kicks in
    directory='/tmp/kt_bayes',
    project_name='cifar10',
    overwrite=True
)

tuner_bayes.search(
    x_train, y_train,
    epochs=15,
    validation_data=(x_val, y_val),
    batch_size=128,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=0
)

print('\nBest hyperparameters (Bayesian):')
best_hp_bayes = tuner_bayes.get_best_hyperparameters(num_trials=1)[0]
for key, val in best_hp_bayes.values.items():
    print(f'  {key}: {val}')

In [ ]:
# ── Compare all three tuners: top-5 trials each ──────────────────────────────
def get_top_accs(tuner, n=5):
    return [tuner.oracle.get_trial(tid).score
            for tid in list(tuner.oracle.trials.keys())[:n]
            if tuner.oracle.get_trial(tid).score is not None]

fig, ax = plt.subplots(figsize=(10, 5))
for tuner, name, c in [(tuner_random,'RandomSearch','#3498db'),
                        (tuner_hb,    'Hyperband',   '#e74c3c'),
                        (tuner_bayes, 'Bayesian',    '#2ecc71')]:
    accs = sorted(get_top_accs(tuner, 10), reverse=True)
    ax.plot(accs, 'o-', label=name, color=c)

ax.set_title('Tuner Comparison — Top Trial Val Accuracies (sorted)')
ax.set_xlabel('Trial rank'); ax.set_ylabel('Val Accuracy')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Train best model from Hyperband to convergence ───────────────────────────
print('=== Training best model from Hyperband to convergence ===')
best_model = tuner_hb.get_best_models(num_models=1)[0]
best_model.build(input_shape=(None, 32, 32, 3))
best_model.summary()

# Re-compile and train longer
best_lr = best_hp_hb.get('learning_rate')
best_model.compile(
    optimizer=keras.optimizers.Adam(best_lr),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

hist_best = best_model.fit(
    x_train, y_train,
    epochs=40, batch_size=128,
    validation_data=(x_test, y_test),
    callbacks=[
        EarlyStopping(patience=8, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)
    ],
    verbose=0
)

_, test_acc = best_model.evaluate(x_test, y_test, verbose=0)
print(f'\nBest tuned model — Test Accuracy: {test_acc:.4f}')

fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].plot(hist_best.history['accuracy'],     label='train')
axes[0].plot(hist_best.history['val_accuracy'], label='val')
axes[0].set_title('Best Model Training')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(hist_best.history['loss'],     label='train')
axes[1].plot(hist_best.history['val_loss'], label='val')
axes[1].set_title('Best Model Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── A/B: Default hyperparams vs Tuned ────────────────────────────────────────
print('=== A/B: Default vs Tuned ===')

# Default model (manually chosen hyperparams)
def build_default():
    inp = keras.Input(shape=(32,32,3))
    x   = data_aug(inp)
    for f in [32, 64]:
        x = layers.Conv2D(f, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(10, activation='softmax')(x)
    m   = keras.Model(inp, out)
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

default_model = build_default()
hist_default  = default_model.fit(
    x_train, y_train, epochs=30, batch_size=128,
    validation_data=(x_test, y_test),
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=0
)
_, default_acc = default_model.evaluate(x_test, y_test, verbose=0)
print(f'Default model test accuracy:  {default_acc:.4f}')
print(f'Tuned model test accuracy:    {test_acc:.4f}')
print(f'Improvement:                  {(test_acc - default_acc)*100:+.2f}%')

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(['Default Hyperparams', 'Keras Tuner (Hyperband)'],
       [default_acc, test_acc], color=['#3498db','#e74c3c'], edgecolor='k', width=0.4)
ax.set_ylim(0.6, 1.0); ax.set_title('Default vs Tuned — Test Accuracy')
for i, v in enumerate([default_acc, test_acc]):
    ax.text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Hyperparameter importance visualisation ───────────────────────────────────
# Extract all trial results from Bayesian tuner
trials = []
for tid, trial in tuner_bayes.oracle.trials.items():
    if trial.score is not None:
        hp_vals = trial.hyperparameters.values.copy()
        hp_vals['val_accuracy'] = trial.score
        trials.append(hp_vals)

import pandas as pd
df = pd.DataFrame(trials)
print('\nAll trial results:')
print(df.sort_values('val_accuracy', ascending=False).to_string(index=False))

# Correlation of numeric HPs with val_accuracy
numeric_cols = df.select_dtypes(include='number').columns.tolist()
corr = df[numeric_cols].corr()['val_accuracy'].drop('val_accuracy').sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
corr.plot(kind='barh', ax=ax, color=['red' if v < 0 else 'green' for v in corr])
ax.axvline(0, color='k', lw=0.8)
ax.set_title('Hyperparameter Correlation with Val Accuracy')
ax.set_xlabel('Pearson r')
plt.tight_layout(); plt.show()